# Notes
Helpful info from udemy course: 
https://github.com/mrdbourke/pytorch-deep-learning/blob/main/01_pytorch_workflow.ipynb

# GitHub Repository:
https://github.com/mpennino/Future_DW_NO3

In [1]:
# Import libraries
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import random


import pyarrow as pa
import pyarrow.parquet as pq


In [2]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [3]:
# Load Observation Dataset
# DATA = readRDS(paste0(strap_dir,'Data/Models/RF_bi_model_All_DATA_all_vars_','Trends_Conc_PWS_GW_05to20', '.rds'))
#future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/StRAPs/StRAP4/SSWR.405.1_Future_DW/Data/'
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'


dataset1 = 'Dataset_RF_Model_SW_COMIDS.csv'
#dataset1 = 'Dataset_RF_Model_GW_COMIDS.csv'

input_data_ = pd.read_csv(future_dir+dataset1)
input_data_.head(3)

,COMID,viol_freq,PopDen2010Ws,PctForest2019Ws,PctCrop2019Ws,precip9120ws,tmean9120ws,BFIWs,permws,RockNWs,N_Surp_kgsqkm_2017ws,ElevWs,Fe2O3Ws,NHDslope_Pct_Ws,Viol_Class
0,12558,0,21.5842,40.60,7.02,446.418385,9.098713,64.9434,7.301968,413.0962,1281.176035,1745.8597,2.7018,9.338324,0
1,12564,0,4.8193,63.36,0.00,454.415442,9.029232,65.0000,8.889206,422.7846,350.849192,1866.1304,3.6657,15.421640,0
2,12606,0,1.1832,68.73,0.00,678.346321,2.958328,69.1535,14.232988,27.3868,492.682789,2905.9906,2.5098,8.917415,0


In [4]:
input_data1 = input_data_.drop(columns=['viol_freq']) 

names_list = input_data1.columns.tolist()
print(names_list) 

['COMID', 'PopDen2010Ws', 'PctForest2019Ws', 'PctCrop2019Ws', 'precip9120ws', 'tmean9120ws', 'BFIWs', 'permws', 'RockNWs', 'N_Surp_kgsqkm_2017ws', 'ElevWs', 'Fe2O3Ws', 'NHDslope_Pct_Ws', 'Viol_Class']


In [5]:
# Remove extra fields
# For Surface Water Dataset (,'AgDrain_pctWs','Hillslope_PctWs','BFIWs')
#input_data = input_data_.drop(columns=['HUC12','viol_freq','PopDen2010Ws','WaterInputWs','wdrw_LDWs','FertWs','CBNFWs','ManureWs','Septic_km2Cat']) 
input_data = input_data_.drop(columns=['COMID','viol_freq']) 

# For Groundwater Dataset
#input_data = input_data_.drop(columns=['HUC12','viol_freq','PopDen2010Cat','AgKffactCat','Septic_km2Cat','AgDrain_pctCat','WaterInputCat','wdrw_LDCat','BFICat','Hillslope_PctCat']) 

input_data.head(3)


,PopDen2010Ws,PctForest2019Ws,PctCrop2019Ws,precip9120ws,tmean9120ws,BFIWs,permws,RockNWs,N_Surp_kgsqkm_2017ws,ElevWs,Fe2O3Ws,NHDslope_Pct_Ws,Viol_Class
0,21.5842,40.60,7.02,446.418385,9.098713,64.9434,7.301968,413.0962,1281.176035,1745.8597,2.7018,9.338324,0
1,4.8193,63.36,0.00,454.415442,9.029232,65.0000,8.889206,422.7846,350.849192,1866.1304,3.6657,15.421640,0
2,1.1832,68.73,0.00,678.346321,2.958328,69.1535,14.232988,27.3868,492.682789,2905.9906,2.5098,8.917415,0


In [6]:
# Calculate accuracy (a classification metric)
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item() # torch.eq() calculates where two tensors are equal
    acc = (correct / len(y_pred)) * 100 
    return acc

# Create Balanced Dataset


In [7]:
print(input_data['Viol_Class'].value_counts())

Viol_Class
0    14904
1       41
Name: count, dtype: int64


In [8]:
# Save Preditor Data for SHAP Analysis
X_input_data = input_data.drop(columns=['Viol_Class']).values
X_input_data.shape


(14945, 12)

In [9]:
# Convert to tensor data
X_input_data = torch.from_numpy(X_input_data).type(torch.float)
X_input_data.shape,X_input_data.dtype

(torch.Size([14945, 12]), torch.float32)

In [10]:
min_size = input_data['Viol_Class'].value_counts().min()
min_size

np.int64(41)

In [11]:
# Find the size of the smallest class
min_size = input_data['Viol_Class'].value_counts().min()

min_size  = min_size * 10

# Sample exactly 'min_size' elements from each binary group
#balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42).reset_index(drop=True)

# If increasing the min_size, you can use the 'replace=True' argument to allow for sampling with replacement
balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42, replace=True).reset_index(drop=True)

print(balanced_df['Viol_Class'].value_counts())

Viol_Class
0    410
1    410
Name: count, dtype: int64


# Transform data to torch tensor


In [12]:
# Convert to tensors and split into train and test sets
from sklearn.model_selection import train_test_split
#X = input_data.drop(columns=['Viol_Class']).values # when use this the model just predicts the majority class
#y = input_data['Viol_Class'].values
X = balanced_df.drop(columns=['Viol_Class']).values
y = balanced_df['Viol_Class'].values

# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# Make a copy to use later
X_full = X.clone()
y_full = y.clone()

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2,
                                                    random_state=2
)

#X_train[:5], y_train[:5]

C:\Users\MPennino\AppData\Local\Temp\ipykernel_108520\3056076170.py:10: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at D:\bld\libtorch_1784990676194\work\torch\csrc\utils\tensor_numpy.cpp:219.)
  y = torch.from_numpy(y).type(torch.float)


In [68]:
# print(y_train.unique(return_counts=True)),
# print(y_test.unique(return_counts=True)),

In [69]:
#X.dtype, y.dtype, X.size(), y.size()

# Create NN Model

In [13]:
nrows = X_train.size()[0]
ncols = X_train.size()[1]
nrows,ncols

(656, 12)

In [71]:
# Build SW model with non-linear activation function
# from torch import nn

# featureNum = 5
# class BinaryClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.layer_1 = nn.Linear(in_features=ncols, out_features=featureNum) 
#         self.layer_2 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_3 = nn.Linear(in_features=featureNum, out_features=1)
#         self.relu = nn.ReLU() # <- add in ReLU activation function (for non-linearity)
#         #self.relu= nn.leakyReLU() # <- add in ReLU activation function (for non-linearity)
#         # Can also put sigmoid in the model 
#         # This would mean you don't need to use it on the predictions
#         # self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#       # Intersperse the ReLU activation function between layers
#        return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))

# model1 = BinaryClassifier().to(device)
# print(model1)

In [72]:
# # Build GW model with non-linear activation function
# from torch import nn

# featureNum = 5
# class BinaryClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # This code works for SW HUC12
#         # self.layer_1 = nn.Linear(in_features=ncols, out_features=5) 
#         # self.layer_2 = nn.Linear(in_features=5, out_features=5)
#         # self.layer_3 = nn.Linear(in_features=5, out_features=1)

#         self.layer_1 = nn.Linear(in_features=ncols, out_features=featureNum) 
#         self.layer_2 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_3 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_4 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_5 = nn.Linear(in_features=featureNum, out_features=1)

#         self.relu = nn.ReLU(0.1) # <- add in ReLU activation function (for non-linearity)
#         #self.relu= nn.leakyReLU() # <- add in ReLU activation function (for non-linearity)
#         # Can also put sigmoid in the model 
#         # This would mean you don't need to use it on the predictions
#         # self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#       # Intersperse the ReLU activation function between layers
#        #return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))
#        #return self.layer_4(self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x))))))
#        return self.layer_5(self.layer_4(self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))))

# model1 = BinaryClassifier().to(device)
#print(model2)

In [14]:
import torch
import torch.nn as nn

class ImprovedBinaryClassifier(nn.Module):
    def __init__(self, input_dim=ncols, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.1),             # Prevents dead neurons
            nn.BatchNorm1d(hidden_dim),     # Stabilizes training
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),                # Prevents overfitting
            
            nn.Linear(hidden_dim, 1)        # Outputs raw logits (No Sigmoid here!)
        )
        
    def forward(self, x):
        return self.network(x)

model1 = ImprovedBinaryClassifier().to(device)


In [15]:
# Setup loss and optimizer 
loss_fn = nn.BCEWithLogitsLoss()
#optimizer = torch.optim.SGD(model1.parameters(), lr=0.01)
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.01)
#optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.01)

# Train the Model

In [16]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, SubsetRandomSampler
from sklearn.model_selection import StratifiedKFold

# Mock data: 1000 samples, 10 features each
# X = np.random.randn(1000, 10).astype(np.float32)
# y = np.random.randint(0, 2, size=1000).astype(np.float32)

# # Convert to PyTorch Tensors
#X_tensor = torch.tensor(X)
#y_tensor = torch.tensor(y).unsqueeze(1) # Shape: [1000, 1] for BCELoss
X_tensor = X.clone().detach()
y_tensor = y.clone().detach().unsqueeze(1) # Shape: [1000, 1] for BCELoss
dataset = TensorDataset(X_tensor, y_tensor)


In [112]:
batchsize = int(len(dataset) * 0.2)
batchsize

164

In [ ]:
# K-fold cross-validation model training
# 22s for 100 epochs, 1m 50s for 500 epochs. (when set batch size large, e.g., 20% of dataset, the time is shorter. )
epochs = 500
k_folds = 5

seedvalue = 18 # 24 for SW
torch.manual_seed(seedvalue)

# Set the random seed for PyTorch (GPU / CUDA) if you use a graphics card
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(seedvalue)

# Set the random seed for NumPy
np.random.seed(seedvalue)

# Set the random seed for Python's built-in random library
random.seed(seedvalue)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seedvalue)

fold_results = []
models = []
val_samplers = [] # save test predictions for each fold for later analysis
val_loaders = [] # save test predictions for each fold for later analysis

# skf.split needs the original X and y to calculate class stratifications
for fold, (train_ids, val_ids) in enumerate(skf.split(X, y)): # val_ids are the indices for the validation (test) set for this fold
    print(f"--- FOLD {fold + 1} ---")
    
    # 1. Create data samplers for this specific fold
    train_sampler = SubsetRandomSampler(train_ids)
    val_sampler = SubsetRandomSampler(val_ids)
    
    # 2. Create DataLoaders
    train_loader = DataLoader(dataset, batch_size=batchsize, sampler=train_sampler, shuffle=False) # typical bastch_size = 32 ()
    val_loader = DataLoader(dataset, batch_size=batchsize, sampler=val_sampler, shuffle=False)
    
    # 3. INITIALIZE A FRESH MODEL AND OPTIMIZER FOR THIS FOLD
    model1 = ImprovedBinaryClassifier()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model1.parameters(), lr=0.001)
    
    # 4. Training Loop for this fold
    model1.train()
    for epoch in range(epochs):
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model1(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
    # 5. Evaluation Loop for this fold
    model1.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model1(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            
            # Convert logits to binary predictions (0 or 1)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == targets).sum().item()
            total += targets.size(0)
            
    # Calculate metrics for this fold
    fold_acc = (correct / total) * 100
    fold_results.append(fold_acc) # store the accuracy for this fold
    models.append(model1) # store the trained model for this fold to use later for predictions or SHAP analysis
    val_samplers.append(val_sampler) # store the validation sampler for this fold
    val_loaders.append(val_loader) # store the validation/test dataset for this fold

    print(f"Fold {fold + 1} Accuracy: {fold_acc:.2f}%\n")

# Overall performance
print(f"Average {k_folds}-Fold Accuracy: {np.mean(fold_results):.2f}%")

--- FOLD 1 ---
Fold 1 Accuracy: 96.95%

--- FOLD 2 ---
Fold 2 Accuracy: 96.34%

--- FOLD 3 ---
Fold 3 Accuracy: 97.56%

--- FOLD 4 ---
Fold 4 Accuracy: 93.29%

--- FOLD 5 ---
Fold 5 Accuracy: 93.29%

Average 5-Fold Accuracy: 95.49%


In [ ]:
val_samplers[0].indices
#sampler_tensor = torch.tensor(val_samplers[0].indices)
#sampler_tensor.shape, sampler_tensor[0:5]

(torch.Size([164]), tensor([ 4,  5, 17, 18, 19]))

In [35]:
type(models), type(tests), type(models[0]), type(val_samplers[0]), type(X_test), type(y_test)

(list,
 list,
 __main__.ImprovedBinaryClassifier,
 torch.utils.data.sampler.SubsetRandomSampler,
 torch.Tensor,
 torch.Tensor)

# Model Evaluation Metrics
*PCC, Sensativity, Specificity, AUC

In [106]:
type(models[0])

__main__.ImprovedBinaryClassifier

In [61]:
len(models), len(val_loaders)

(5, 5)

In [97]:
pcc_all = []
sensitivity_all = []
specificity_all = []
with torch.no_grad():
    
    for i in range(len(models)):
        model = models[i]
        model.eval()  # Set the model to evaluation mode
        val_loader = val_loaders[i]
        val_loss = 0.0
        correct = 0
        total = 0
        # Convert the dataloader into a Python iterator
        data_iterator = iter(val_loader)

        # Grab the first mini-batch
        inputs, targets = next(data_iterator)

        #for inputs, targets in val_loader:  # Using the ith validation loader
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        val_loss += loss.item() * inputs.size(0)
        
        # Convert logits to binary predictions (0 or 1)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        # Calculate True Positives, True Negatives, False Positives, False Negatives
        TP = torch.sum((preds == 1) & (targets == 1)).float()
        TN = torch.sum((preds == 0) & (targets == 0)).float()
        FP = torch.sum((preds == 1) & (targets == 0)).float()
        FN = torch.sum((preds == 0) & (targets == 1)).float()
        PCC = (TP + TN) / (TP + TN + FP + FN)
        sensitivity = TP / (TP + FN )
        specificity = TN / (TN + FP )
        #print(PCC, sensitivity, specificity)
        pcc_all.append(PCC)
        sensitivity_all.append(sensitivity)
        specificity_all.append(specificity)
        #print(len(preds), len(targets), len(inputs))

pcc_mean = np.mean(pcc_all)
sensitivity_mean = np.mean(sensitivity_all)
specificity_mean = np.mean(specificity_all)


#fold_acc = (sum(correct_all) / sum(total_all)) * 100
#fold_acc
#print(pcc_all)
print(f"PCC: {pcc_mean:.4f}")
print(f"Sensitivity (True Positives): {sensitivity_mean:.4f}")
print(f"Specificity (True Negatives): {specificity_mean:.4f}")

PCC: 0.9549
Sensitivity (True Positives): 1.0000
Specificity (True Negatives): 0.9098


In [68]:
len(pcc_all), len(sensitivity_all), len(specificity_all)

(5, 5, 5)

In [102]:
# AUC Calculation
from sklearn.metrics import roc_auc_score
import numpy as np

auc_scores = []
with torch.no_grad():
    for i in range(len(models)):
        model = models[i]
        model.eval()  # Set the model to evaluation mode
        val_loader = val_loaders[i]
        val_loss = 0.0
        correct = 0
        total = 0
        # Convert the dataloader into a Python iterator
        data_iterator = iter(val_loader)

        # Grab the first mini-batch
        inputs, targets = next(data_iterator)
        
        with torch.inference_mode():
            preds = torch.round(torch.sigmoid(model(inputs))).squeeze()

            preds2 = preds.detach().cpu().tolist()
            target2 = targets.detach().cpu().tolist()


            # 2. Calculate the AUC Score
            auc_score = roc_auc_score(target2, preds2)
            auc_scores.append(auc_score)

auc_mean = np.mean(auc_scores)
print(f"Test AUC: {auc_mean:.4f}")

Test AUC: 0.9549


# Save the Model

In [103]:
model_dir = future_dir + "/Models"
model_dir

'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3//Models'

In [105]:
# Path
modelpath = "/torch_models_5fold_future_NO3_sw.pth"

# 1. Create a dictionary containing the state_dict of each model
models_checkpoint = {
    f"model_{i}": model.state_dict() for i, model in enumerate(models)
}

# 2. Save the dictionary to a single file (.pt or .pth extension)
torch.save(models_checkpoint, model_dir + modelpath)

# Save the test datasets

In [117]:
import pickle

# Extract state/parameters or pickle a list of datasets if serializable
dataset_list = [loader.dataset for loader in val_loaders]
len(dataset_list), type(dataset_list[0]), dataset_list[0].tensors[0].shape, dataset_list[0].tensors[1].shape

testnames = "/test_datasets_sw.pkl"
with open(model_dir + testnames, "wb") as f:
    pickle.dump(dataset_list, f)